<a href="https://colab.research.google.com/github/CodeBreaker-0111/CodeBreaker-0111/blob/main/Celebal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
import xgboost as xgb
import lightgbm as lgb


In [ ]:
train = pd.read_parquet('train.parquet')
test  = pd.read_parquet('test.parquet')
train['target'] = train['target'].astype(int)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTarget distribution:")
print(train['target'].value_counts())

Train shape: (1639424, 7)
Test shape: (409856, 7)

Target distribution:
target
0    1625386
1      14038
Name: count, dtype: int64


In [ ]:
print("\n--- EDA ---")
print(train.describe())
print("\nMissing values:", train.isnull().sum().sum())

anomaly = train[train['target']==1]
normal  = train[train['target']==0]
print("\nAnomaly X1 mean:", anomaly['X1'].mean(), "| Normal X1 mean:", normal['X1'].mean())
print("Anomaly X5 mean:", anomaly['X5'].mean(), "| Normal X5 mean:", normal['X5'].mean())


--- EDA ---
                                Date            X1            X2  \
count                        1639424  1.639424e+06  1.639424e+06   
mean   2022-12-03 07:23:43.817145600  1.139258e+00  5.488189e+00   
min              2020-12-16 00:00:00  1.000000e+00  5.412539e+00   
25%              2021-12-10 00:00:00  1.049171e+00  5.480597e+00   
50%              2022-11-30 00:00:00  1.105171e+00  5.488979e+00   
75%              2023-11-23 00:00:00  1.214096e+00  5.496717e+00   
max              2024-12-11 00:00:00  4.014850e+00  5.541852e+00   
std                              NaN  1.391992e-01  1.342811e-02   

                 X3            X4            X5        target  
count  1.639424e+06  1.639424e+06  1.639424e+06  1.639424e+06  
mean   4.110388e+32  2.706323e+29  1.187219e+00  8.562764e-03  
min    1.000000e+00  1.000000e+00  0.000000e+00  0.000000e+00  
25%    1.000000e+00  1.000000e+00  0.000000e+00  0.000000e+00  
50%    1.000000e+00  1.000000e+00  6.931472e-01  0.000

In [ ]:
def add_features(df):
    df = df.copy()
    # Time features
    df['month']     = df['Date'].dt.month
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['quarter']   = df['Date'].dt.quarter

    # Log transform (X3, X4 are exponentially scaled)
    df['log_X3']  = np.log1p(df['X3'])
    df['log_X4']  = np.log1p(df['X4'])

    # Deviation features
    df['X1_dev']  = df['X1'] - 1.0          # baseline deviation
    df['X2_dev']  = df['X2'] - df['X2'].mean() if 'target' not in df.columns else df['X2'] - 5.488
    df['X5_high'] = (df['X5'] > 2.8).astype(int)  # high X5 = anomaly indicator

    # Interaction
    df['logdiff'] = df['log_X3'] - df['log_X4']
    df['X1_X2']   = df['X1'] * df['X2']

    return df

train = add_features(train)
test  = add_features(test)

FEATURES = ['X1','X2','X3','X4','X5',
            'month','dayofweek','quarter',
            'log_X3','log_X4','X1_dev','X2_dev',
            'X5_high','logdiff','X1_X2']

X  = train[FEATURES].values
y  = train['target'].values
Xt = test[FEATURES].values

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\nTrain size: {X_tr.shape[0]} | Val size: {X_val.shape[0]}")
print(f"Anomaly in val: {y_val.sum()} / {len(y_val)}")



Train size: 1311539 | Val size: 327885
Anomaly in val: 2808 / 327885


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_val_sc = scaler.transform(X_val)

print("\n--- Logistic Regression ---")
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_tr_sc, y_tr)
lr_pred = lr.predict(X_val_sc)
print(f"F1 Macro: {f1_score(y_val, lr_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_val, lr_pred):.4f}")
print(classification_report(y_val, lr_pred))


--- Logistic Regression ---
F1 Macro: 0.6074
Accuracy: 0.9500
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    325077
           1       0.14      0.93      0.24      2808

    accuracy                           0.95    327885
   macro avg       0.57      0.94      0.61    327885
weighted avg       0.99      0.95      0.97    327885



In [ ]:
scale_pos = float((y==0).sum()) / float((y==1).sum())
print(f"\n--- XGBoost (scale_pos_weight={scale_pos:.0f}) ---")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
xgb_pred = xgb_model.predict(X_val)
print(f"F1 Macro: {f1_score(y_val, xgb_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_val, xgb_pred):.4f}")
print(classification_report(y_val, xgb_pred))


--- XGBoost (scale_pos_weight=116) ---
F1 Macro: 0.6296
Accuracy: 0.9588
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    325077
           1       0.16      0.94      0.28      2808

    accuracy                           0.96    327885
   macro avg       0.58      0.95      0.63    327885
weighted avg       0.99      0.96      0.97    327885



In [ ]:
print("\n--- LightGBM ---")
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
lgb_pred = lgb_model.predict(X_val)
print(f"F1 Macro: {f1_score(y_val, lgb_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_val, lgb_pred):.4f}")
print(classification_report(y_val, lgb_pred))


--- LightGBM ---
F1 Macro: 0.6134
Accuracy: 0.9520
              precision    recall  f1-score   support

           0       1.00      0.95      0.98    325077
           1       0.15      0.94      0.25      2808

    accuracy                           0.95    327885
   macro avg       0.57      0.95      0.61    327885
weighted avg       0.99      0.95      0.97    327885



In [ ]:
print("\n--- Ensemble (XGB + LGB) ---")
xgb_prob = xgb_model.predict_proba(X_val)[:, 1]
lgb_prob  = lgb_model.predict_proba(X_val)[:, 1]
ens_prob  = 0.5 * xgb_prob + 0.5 * lgb_prob

# Find best threshold for F1
best_f1, best_thresh = 0, 0.5
for t in np.arange(0.1, 0.9, 0.05):
    preds_t = (ens_prob >= t).astype(int)
    f = f1_score(y_val, preds_t, average='macro')
    if f > best_f1:
        best_f1, best_thresh = f, t

print(f"Best threshold: {best_thresh:.2f} | Best F1: {best_f1:.4f}")
final_val_pred = (ens_prob >= best_thresh).astype(int)
print(classification_report(y_val, final_val_pred))


--- Ensemble (XGB + LGB) ---
Best threshold: 0.85 | Best F1: 0.7388
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    325077
           1       0.34      0.87      0.49      2808

    accuracy                           0.98    327885
   macro avg       0.67      0.93      0.74    327885
weighted avg       0.99      0.98      0.99    327885



In [ ]:
print("\n--- Training final models on full data ---")
xgb_model.fit(X, y)
lgb_model.fit(X, y)

xgb_test_prob = xgb_model.predict_proba(Xt)[:, 1]
lgb_test_prob  = lgb_model.predict_proba(Xt)[:, 1]
test_ens_prob  = 0.5 * xgb_test_prob + 0.5 * lgb_test_prob

final_preds = (test_ens_prob >= best_thresh).astype(int)
print("Test prediction distribution:", np.bincount(final_preds))


--- Training final models on full data ---
Test prediction distribution: [400889   8967]


In [ ]:
submission = pd.DataFrame({'ID': test['ID'], 'target': final_preds})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved!")
print(submission.head(10))
print(submission['target'].value_counts())


Submission saved!
   ID  target
0   0       0
1   1       0
2   2       0
3   3       0
4   4       0
5   5       0
6   6       0
7   7       0
8   8       1
9   9       0
target
0    400889
1      8967
Name: count, dtype: int64
